# 03 · Semantic Retrieval with ChromaDB
**Goal:** store small documents, search by meaning, filter metadata, and persist the collection.

This notebook uses a new collection name and a small campus-help dataset.

## Install

In [ ]:
%pip install -q chromadb sentence-transformers

## Create a collection and add records

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./local_vector_data")
collection = client.get_or_create_collection(name="campus_help_v1")

records = [
    ("library_hours", "The campus library is open from 8 AM to 10 PM on weekdays.", {"topic": "facilities"}),
    ("wifi_help", "Connect to the student Wi-Fi using your campus credentials.", {"topic": "IT"}),
    ("exam_rules", "Bring your student identity card to every examination.", {"topic": "academics"}),
    ("sports_center", "The sports center has badminton courts and a fitness room.", {"topic": "sports"}),
    ("password_reset", "If you forget your portal password, use the account recovery page.", {"topic": "IT"}),
]
collection.upsert(
    ids=[r[0] for r in records],
    documents=[r[1] for r in records],
    metadatas=[r[2] for r in records],
)
print("Stored records:", collection.count())

## Search using natural language

In [ ]:
result = collection.query(
    query_texts=["I cannot sign into my student account"],
    n_results=3,
)
for doc, meta, distance in zip(
    result["documents"][0], result["metadatas"][0], result["distances"][0]
):
    print(f"distance={distance:.4f} | {meta['topic']} | {doc}")

## Metadata filtering

In [ ]:
filtered = collection.query(
    query_texts=["Where can I play badminton?"],
    where={"topic": "sports"},
    n_results=2,
)
print(filtered["documents"][0])

## Inspect, update, and delete

In [ ]:
print("IDs:", collection.get()["ids"])
collection.update(
    ids=["library_hours"],
    documents=["The campus library operates from 7:30 AM to 9:30 PM on weekdays."],
    metadatas=[{"topic": "facilities", "verified": True}],
)
print("Updated:", collection.get(ids=["library_hours"])["documents"])

# Demonstrate deletion on a disposable sample record.
collection.upsert(ids=["temporary_notice"], documents=["Temporary maintenance notice."],
                  metadatas=[{"topic": "temporary"}])
collection.delete(ids=["temporary_notice"])
print("Temporary record remaining:", collection.get(ids=["temporary_notice"])["ids"])

## Notes and exercises
- Chroma's default embedding function may download a model depending on your environment/version.
- The persistent directory `local_vector_data/` is created beside the notebook. Remove it only when you want to reset this learning database.
- Add a new topic, try different natural-language questions, and compare retrieved results.
- A lower distance generally means a closer match for the collection's configured distance metric; confirm metric semantics for your setup.